01_url_preprocess.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1hgzWX8MbZ2Xd3rjl-e-dj3dqvJf2u78S

# DeepTrace — Module 1: URL Phishing
## Notebook 1 of 2: Preprocessing
### Runtime: CPU (default) | Time: ~10 min

**Steps:**
1. Install deps
2. Upload kaggle.json when asked
3. Downloads PhiUSIIL (235K URLs) + live feeds
4. Extracts 52 URL features
5. Saves + downloads `url_processed.zip`

In [ ]:
# CELL 1 — Install
!pip install -q kaggle tldextract scikit-learn pandas numpy xgboost lightgbm joblib imbalanced-learn tqdm

In [ ]:
# CELL 2 — Kaggle credentials
from google.colab import files
import os

print('Upload your kaggle.json')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
fname = next(iter(uploaded.keys()))
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded[fname])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✓ Kaggle credentials set')

In [ ]:
# CELL 3 — Download datasets
import os

os.makedirs('/content/data/raw', exist_ok=True)

print('Downloading PhiUSIIL...')
!kaggle datasets download -d harisudhan411/phishing-and-legitimate-urls --unzip -p /content/data/raw/ -q
print('✓ PhiUSIIL downloaded')

import urllib.request
for url, out in [
    ('https://data.phishtank.com/data/online-valid.csv', '/content/data/raw/phishtank_live.csv'),
    ('https://openphish.com/feed.txt', '/content/data/raw/openphish_live.txt'),
]:
    try:
        urllib.request.urlretrieve(url, out)
        print(f'✓ {os.path.basename(out)} downloaded')
    except Exception as e:
        print(f'⚠ {os.path.basename(out)} skipped: {e}')

print('\nRaw files:')
!ls -lh /content/data/raw/

In [ ]:
# CELL 4 — Inspect downloaded files
import pandas as pd, glob

for f in glob.glob('/content/data/raw/**/*.csv', recursive=True):
    try:
        df = pd.read_csv(f, nrows=3, low_memory=False)
        print(f'{f}: shape needs full load, cols={list(df.columns)}')
    except Exception as e:
        print(f'⚠ {f}: {e}')

In [ ]:
# CELL 5 — Feature extraction (52 features, fully safe)
import re, math, tldextract
from urllib.parse import urlparse

def entropy(s):
    s = str(s)
    if not s: return 0.0
    freq = {c: s.count(c)/len(s) for c in set(s)}
    return -sum(p*math.log2(p) for p in freq.values())

def safe_port(parsed):
    try:
        p = parsed.port
        return int(p is not None and p not in [80,443])
    except: return 0

def extract_url_features(url):
    try:
        url = str(url).strip()
        if not url or url.lower() in ['nan','none','null','']: return [0.0]*52
        safe = url if url.startswith(('http://','https://')) else 'http://'+url
        parsed = urlparse(safe)
        try:
            ext = tldextract.extract(safe)
            sub, dom, suf = ext.subdomain or '', ext.domain or '', ext.suffix or ''
        except:
            sub, dom, suf = '', '', ''
        path  = parsed.path  or ''
        query = parsed.query or ''
        nloc  = parsed.netloc or ''
        ul    = url.lower()

        return [
            len(url), len(dom), len(sub), len(path), len(query), len(nloc),
            url.count('.'), url.count('-'), url.count('_'), url.count('/'),
            url.count('?'), url.count('='), url.count('@'), url.count('!'),
            url.count('%'), url.count('+'), url.count('~'), url.count(','),
            url.count('*'), url.count('#'), url.count('$'), url.count('&'),
            sum(c.isdigit() for c in url)/max(len(url),1),
            sum(c.isalpha() for c in url)/max(len(url),1),
            entropy(dom), entropy(url),
            int(bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', url))),
            int('@' in url),
            int('//' in path),
            int(url.startswith('https')),
            int(bool(sub and sub!='www')),
            int(len(sub.split('.'))>2),
            int(bool(re.search(r'secure|account|update|login|verify|bank|paypal|ebay|amazon', ul))),
            int(bool(re.search(r'free|click|win|prize|offer|discount|bonus', ul))),
            int(bool(re.search(r'\.exe|\.zip|\.rar|\.scr|\.bat', ul))),
            int('-' in dom),
            int(bool(re.search(r'\d', dom))),
            path.count('/'),
            len(path.split('/')[-1]),
            int(bool(re.search(r'wp-admin|phishing|hack|malware', path.lower()))),
            len(query.split('&')) if query else 0,
            len(suf),
            int(suf in ['tk','ml','ga','cf','gq','xyz','top','click','link','pw']),
            safe_port(parsed),
            len(re.findall(r'\d+', dom)),
            int(len(dom)>0 and max(dom.count(c) for c in set(dom))/len(dom)>0.4),
            int(bool(re.search(r'%[0-9a-fA-F]{2}', url))),
            int(bool(re.search(r'google|facebook|apple|microsoft|netflix|instagram|twitter|linkedin', ul) and
                not re.search(r'\.(google|facebook|apple|microsoft|netflix|instagram|twitter|linkedin)\.com$', nloc.lower()))),
            int(bool(re.search(r'bit\.ly|tinyurl|t\.co|goo\.gl|ow\.ly', ul))),
            int('xn--' in ul),
            len(sub.split('.')) if sub else 0,
            int(bool(parsed.fragment)),
        ]
    except:
        return [0.0]*52

FEATURE_NAMES = [
    'url_length','domain_length','subdomain_length','path_length','query_length','netloc_length',
    'dot_count','hyphen_count','underscore_count','slash_count','question_count','equals_count',
    'at_count','exclaim_count','percent_count','plus_count','tilde_count','comma_count',
    'asterisk_count','hash_count','dollar_count','ampersand_count',
    'digit_ratio','alpha_ratio','domain_entropy','url_entropy',
    'has_ip','has_at','has_double_slash','is_https','has_non_www_subdomain','deep_subdomain',
    'suspicious_keywords','spam_keywords','executable_extension',
    'hyphen_in_domain','digit_in_domain',
    'path_slash_count','filename_length','suspicious_path',
    'num_query_params','tld_length','suspicious_tld',
    'non_standard_port','digit_groups_in_domain','char_repetition',
    'hex_encoded','brand_impersonation','url_shortener','punycode','subdomain_count','has_fragment'
]
assert len(FEATURE_NAMES)==52, f'Got {len(FEATURE_NAMES)}'
# Quick test
t = extract_url_features('http://192.168.1.1/paypal/login?user=abc@evil.tk')
assert len(t)==52
print(f'✓ Feature extraction OK — 52 features')

In [ ]:
# CELL 6 — Load + unify all datasets
import pandas as pd, numpy as np, glob, os
from collections import Counter

all_urls = []

# Primary CSVs
for f in glob.glob('/content/data/raw/**/*.csv', recursive=True):
    try:
        df = pd.read_csv(f, low_memory=False)
        df.columns = df.columns.str.lower().str.strip()
        url_col   = next((c for c in df.columns if 'url' in c), None)
        label_col = next((c for c in df.columns if c in ['label','type','status','phishing','class']), None)
        if url_col and label_col:
            sub = df[[url_col,label_col]].dropna().copy()
            sub.columns = ['url','label']
            sub['label'] = sub['label'].astype(str).str.lower()
            sub['label'] = sub['label'].map(
                lambda x: 1 if x in ['1','phishing','phish','bad','malicious'] else 0)
            all_urls.append(sub)
            print(f'✓ {os.path.basename(f)}: {len(sub)} rows, phishing={sub.label.sum()}')
    except Exception as e:
        print(f'⚠ {os.path.basename(f)}: {e}')

# PhishTank live
if os.path.exists('/content/data/raw/phishtank_live.csv'):
    try:
        pt = pd.read_csv('/content/data/raw/phishtank_live.csv', low_memory=False)
        pt.columns = pt.columns.str.lower()
        uc = 'url' if 'url' in pt.columns else pt.columns[0]
        pt2 = pt[[uc]].dropna().copy()
        pt2.columns = ['url']; pt2['label'] = 1
        all_urls.append(pt2)
        print(f'✓ PhishTank: {len(pt2)} phishing URLs')
    except Exception as e:
        print(f'⚠ PhishTank: {e}')

# OpenPhish live
if os.path.exists('/content/data/raw/openphish_live.txt'):
    with open('/content/data/raw/openphish_live.txt') as f:
        lines = [l.strip() for l in f if l.strip()]
    op = pd.DataFrame({'url': lines, 'label': 1})
    all_urls.append(op)
    print(f'✓ OpenPhish: {len(op)} phishing URLs')

combined = pd.concat(all_urls, ignore_index=True)
combined = combined.drop_duplicates(subset='url').dropna(subset=['url'])
combined['url'] = combined['url'].astype(str).str.strip()
combined = combined[combined['url'].str.len() > 3]
print(f'\nTotal URLs: {len(combined)}')
print(f'Label dist: {Counter(combined.label)}')

In [ ]:
# CELL 7 — Extract features (parallel)
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from tqdm.auto import tqdm

print(f'Extracting features for {len(combined)} URLs...')
urls_list = combined['url'].tolist()

with ThreadPoolExecutor(max_workers=4) as ex:
    features = list(tqdm(ex.map(extract_url_features, urls_list), total=len(urls_list)))

X = np.array(features, dtype=np.float32)
y = combined['label'].values.astype(int)

print(f'Feature matrix: {X.shape}')
print(f'NaN: {np.isnan(X).sum()}  Inf: {np.isinf(X).sum()}')
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
print(f'Label dist: {Counter(y)}')

In [ ]:
# CELL 8 — Split THEN balance (correct order)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter

print('Class dist:', Counter(y))

# Split first — never let test/val see SMOTE data
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.1765, random_state=42, stratify=y_tr)

print(f'Before SMOTE — Train: {Counter(y_tr)}')
print(f'Val: {Counter(y_va)}  Test: {Counter(y_te)}')

# SMOTE only on training set
counts = Counter(y_tr)
ratio  = min(counts.values()) / max(counts.values())
if ratio < 0.4:
    print('Applying SMOTE on train only...')
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_tr, y_tr = smote.fit_resample(X_tr, y_tr)
    print(f'After SMOTE: {Counter(y_tr)}')

print(f'\nFinal shapes:')
print(f'X_tr:{X_tr.shape} X_va:{X_va.shape} X_te:{X_te.shape}')

In [ ]:
# CELL 9 — Save processed data
import numpy as np, json, os

os.makedirs('/content/data/processed', exist_ok=True)
np.save('/content/data/processed/X_train.npy', X_tr)
np.save('/content/data/processed/X_val.npy',   X_va)
np.save('/content/data/processed/X_test.npy',  X_te)
np.save('/content/data/processed/y_train.npy', y_tr)
np.save('/content/data/processed/y_val.npy',   y_va)
np.save('/content/data/processed/y_test.npy',  y_te)
with open('/content/data/processed/feature_names.json','w') as f:
    json.dump(FEATURE_NAMES, f)

print('✓ Saved')
!ls -lh /content/data/processed/

In [ ]:
# CELL 10 — Download
import shutil
from google.colab import files

shutil.make_archive('/content/url_processed','zip','/content/data/processed')
files.download('/content/url_processed.zip')
print('✅ Done — upload this zip to 02_url_train.ipynb')